In [12]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import tqdm

In [2]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()
print(words.__len__())
print(len(words))
if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")


random.shuffle(words)
print(words[:10])
len(words)

32033
32033
Using device: mps
['kylar', 'makyra', 'emmeric', 'jhavia', 'bryelle', 'shane', 'samatar', 'daishon', 'dayten', 'sani']


32033

In [3]:
block_size = 7
epochs = 200
lr = 0.001

In [4]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [5]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [6]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 7]) torch.Size([228146])
torch.Size([205331, 7]) torch.Size([205331])
torch.Size([22815, 7]) torch.Size([22815])


In [7]:
train_ds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())

train_dl = DataLoader(
    dataset = train_ds,
    batch_size = 1024,
    # num_workers = 2,
    # prefetch_factor = 8,
    pin_memory = False,
    shuffle = True,
    # persistent_workers = True,
)

In [8]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.C = torch.nn.Parameter(torch.randn((27, 10)))
        self.W1 = torch.nn.Parameter(torch.randn((70, 200)))
        self.b1 = torch.nn.Parameter(torch.randn(200))
        self.W2 = torch.nn.Parameter(torch.randn((200, 27)))
        self.b2 = torch.nn.Parameter(torch.randn(27))

    def forward(self, Xb):
        emb = self.C[Xb]
        emb = emb.view(-1, 70)
        h = torch.tanh(emb @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        return logits


In [9]:
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9999)



In [11]:
for e in range(epochs):
    for Xb, Yb in train_dl:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    if e % 10 == 0:
        print(e, loss.item(), scheduler.get_last_lr()[0])


0 7.733318328857422 0.0009605953695516413
10 3.8033287525177 0.0007856749978495249
20 2.8093016147613525 0.0006426068892399212
30 2.5218613147735596 0.0005255908807444277
40 2.4141433238983154 0.0004298829946383681
50 2.2431676387786865 0.00035160311156370806
60 2.196368932723999 0.00028757766555822666
70 2.126854181289673 0.00023521098365744832
80 2.2726967334747314 0.0001923800540132795
90 2.2177703380584717 0.00015734845629510286
100 2.2180004119873047 0.00012869596500239504
110 2.195286273956299 0.00010526097171766944
120 2.269247531890869 8.609339202469795e-05
130 2.2684314250946045 7.041614787861669e-05
140 2.178232192993164 5.759366387423519e-05
150 2.185899496078491 4.710610021122256e-05
160 2.1082992553710938 3.8528277727828725e-05
170 2.263226270675659 3.1512440597216933e-05
180 2.109849452972412 2.5774157864208506e-05
190 2.1850883960723877 2.108079225281617e-05


In [ ]:
ix = torch.randint(0, Xdev.shape[0], (2000,), device=device)
Xb, Yb = Xdev[ix], Ydev[ix]
logits = model(Xb)
loss = F.cross_entropy(logits, Yb)
print(loss.item())

2.182788610458374


In [14]:
for i in range(10):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos(ix))
        if ix == 0:
            break
    print(''.join(out))

aylee.
sanc.
raidee.
aritzik.
elisiah.
alesha.
dashian.
falen.
erton.
lojaad.
